# freMTPL2 claim severity (W11)

Welcome back to the glamorous world of motor insurance. This week we take the French motor third-party liability benchmark one step further: from **claim frequency** to **claim severity**. Claim severity is defined as the expected claim cost for a given accident. If you did not attend the last insurance workshop, do not worry: you do not need it in order to work with this one.

The official workshop dataset is a policy-level file derived from the freMTPL2 frequency and severity datasets from [CASdatasets](https://dutangc.github.io/CASdatasets/). Each row contains policy risk characteristics together with:

- `ClaimNb`: the number of claims attached to the policy
- `TotalClaimCost`: the total claim amount across those claims

The available columns are:

- `IDpol`: policy identifier
- `Area`: discretized population-density area, from rural to urban
- `VehPower`: vehicle power class
- `VehAge`: vehicle age in years
- `DrivAge`: driver age in years
- `BonusMalus`: the French bonus-malus risk score
- `VehBrand`: vehicle brand code
- `VehGas`: fuel type
- `Density`: local population density
- `Region`: region in France
- `ClaimNb`: number of claims linked to the policy
- `TotalClaimCost`: total claim cost linked to the policy

A very important nuance: this is **policy-level** severity data, not pure claim-level severity data. If `ClaimNb > 1`, then `TotalClaimCost` is the aggregate cost across several claims. That means there are at least two reasonable targets to think about:

- total cost conditional on having at least one claim
- average cost per claim, for example `TotalClaimCost / ClaimNb`

For this workshop, one thing you will want to do is restrict to rows with `ClaimNb > 0`, since severity modeling is usually conditional on a claim actually happening.

Below you'll find some possible starting points. Pick the level that best suits you, dig in, or ignore them and do your own thing. Happy coding!

##### **Beginner**

- Start by getting oriented with `raw.shape`, `df.shape`, `.head()`, `.info()`, and `.describe()`.
  - How many policy rows are there in total?
  - How many remain after filtering to `ClaimNb > 0`?
  - How many of the positive rows have more than one claim?

- Study the target:
  - Plot a histogram of `TotalClaimCost`.
  - Then plot `np.log1p(TotalClaimCost)` and compare.
  - What do the median, 90th, 95th, and 99th percentiles look like?

- Create a `AvgClaimCost` column:
  - Define `AvgClaimCost = TotalClaimCost / ClaimNb`.
  - Compare the distribution of `AvgClaimCost` with `TotalClaimCost`. What changes?

- Sanity-check the business meaning:
  - Which rows have the very largest `TotalClaimCost` values?
  - Are the biggest totals mostly driven by many claims, one huge claim, or both?

- Explore basic group differences:
  - Compare severity by `VehGas`, `Area`, and `Region`.
  - Is the mean telling the same story as the median?


##### **Intermediate**

- Bucket continuous features (severity models often benefit from grouping):
  - Use `pd.cut()` to create:
    - `DrivAgeGroup` (for example: 18–25, 26–35, 36–50, 51–70, 70+)
    - `VehAgeGroup` (for example: 0–2, 3–10, 11+)
    - Possibly a log-based binning of `Density`.

- Watch out for multi-claim policies:
  - Compute summary statistics of `ClaimNb` among rows with `ClaimNb > 0`.
  - What fraction of severity rows correspond to exactly one claim vs multiple claims?
  - Does `AvgClaimCost` differ systematically between single-claim and multi-claim policies?

- Build a few severity summary tables:
  - Compute mean (and optionally median) severity by:
    - `Region`
    - `VehBrand`
    - `VehPower`
  - Do the rankings change if you use the median instead of the mean?


##### **Advanced**

- Fit a classical severity model:
  - Restrict to rows to `ClaimNb > 0`.
  - Fit a **Gamma GLM with log link** for `TotalClaimCost` offsetting `np.log(ClaimNb)`, or targeting `AvgClaimCost` and using `ClaimNb` as weights.
  - Interpret the strongest coefficients.

- Try a nonlinear model:
  - Train a boosting model (e.g. LightGBM/XGBoost/CatBoost). You may want test using a log-transformed severity target!
  - Compare its predictive performance with the GLM, which works the best?

- Interpret the model:
  - For GLMs: inspect coefficients.
  - For boosting models: examine feature importance or SHAP values.


In [17]:
import pandas as pd

df = pd.read_csv('freMTPL2_severity_policy_level.csv')
df.head()


,IDpol,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ClaimNb,TotalClaimCost
0,1,D,5,0,55,50,B12,Regular,1217,Rhone-Alpes,0,0.0
1,3,D,5,0,55,50,B12,Regular,1217,Rhone-Alpes,0,0.0
2,5,B,6,2,52,50,B12,Diesel,54,Picardie,0,0.0
3,10,B,7,0,46,50,B12,Diesel,76,Aquitaine,0,0.0
4,11,B,7,0,46,50,B12,Diesel,76,Aquitaine,0,0.0
